<a href="https://colab.research.google.com/github/vijaysingh60616-sudo/vijaysingh60616-sudo.github.io/blob/main/RegularizATION_(L1%2C_L2%2C_ElasticNet).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Mathematical FormulaLasso Regression ka cost function (objective) yeh hota hai:$$J(w) = \frac{1}{2n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{d} \vert{}w_j\vert{}$$Kyunki $\vert{}w_j\vert{}$ (absolute value) $w_j = 0$ par differentiable nahi hota, hum aam Gradient Descent ka asani se use nahi kar sakte. Iski jagah hum Soft Thresholding ka istemal karte hain.Agar hum baaki saare weights ko fixed rakh kar sirf ek weight $w_j$ ko optimize karein, toh update rule banta hai:$$w_j = \text{SoftThreshold}(\rho_j, \alpha) = \begin{cases} \rho_j - \alpha & \text{if } \rho_j > \alpha \\ \rho_j + \alpha & \text{if } \rho_j < -\alpha \\ 0 & \text{if } -\alpha \le \rho_j \le \alpha \end{cases}$$Jahan $\rho_j$ ek term hai jo yeh batata hai ki $j$-th feature model ke residual error (jab feature $j$ hata diya jaye) se kitna correlated hai. Yahi property Lasso ko un-important weights ko exactly $0$ karne me madad karti hai (Feature Selection).

In [ ]:
import numpy as np

def soft_threshold(rho, alpha):
    """Soft thresholding operator for Lasso."""
    if rho < -alpha:
        return rho + alpha
    elif rho > alpha:
        return rho - alpha
    else:
        return 0

def coordinate_descent_lasso(X, y, alpha=0.1, num_iters=100):
    m, n = X.shape
    # Weights ko zero se initialize karte hain
    w = np.zeros(n)

    for iteration in range(num_iters):
        for j in range(n):
            # Feature j ke bina predictions calculate karte hain
            X_j = X[:, j]
            y_pred_without_j = X @ w - (w[j] * X_j)

            # Correlation aur Normalizer calculate karte hain
            rho_j = X_j.T @ (y - y_pred_without_j)
            z_j = np.sum(X_j ** 2)

            # Weight ko soft thresholding se update karte hain
            if z_j == 0:
                w[j] = 0
            else:
                # alpha ko m (number of samples) se scale kiya gaya hai
                w[j] = soft_threshold(rho_j, alpha * m) / z_j

    return w

# 1. Synthetic Data generate karte hain
np.random.seed(42)
X = np.random.randn(100, 3)

# Asli weights hain [3, -2, 0] (Teesra feature useless hai)
true_w = np.array([3, -2, 0])
y = X @ true_w + np.random.randn(100) * 0.5

# 2. Coordinate Descent run karte hain
alpha_val = 0.5
w_lasso = coordinate_descent_lasso(X, y, alpha=alpha_val, num_iters=50)

print("True Weights:", true_w)
print(f"Lasso Weights (alpha={alpha_val}):", np.round(w_lasso, 3))

To understand Lasso visually, the best graph to plot is the Lasso Regularization Path. This graph shows exactly how changing the penalty strength ($\alpha$) affects each feature's weight.Since you already have the coordinate_descent_lasso function, X, and y loaded in your Colab notebook, you can just paste this new code block into the next cell to visualize it:

In [ ]:
import matplotlib.pyplot as plt
alphas = np.linspace(0.01, 3.0, 100)
weights_path = []

for a in alphas:
  w = coordinate_descent_lasso(X, y, alpha=a, num_iters=50)
  weights_path.append(w.copy())
weights_path = np.array(weights_path)


plt.figure(figsize=(10,6))


# plot a line foe each feature weight
for j in range(weights_path.shape[1]):
  plt.plot(alphas, weights_path[:, j], linewidth=2,
           label =f'Feature {j+1} (True W: {true_w[j]})')



plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.title("Lasso Regularization Path(Alpha vs. Weights)", fontsize=14 )
plt.xlabel("Alpha(penalty Strength)", fontsize=12)
plt.ylabel("Cofficent/ Weight VAlue", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()




# **L2  Rerularization (Ridge Regresion)**
$$J(w) = \frac{1}{2n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \frac{\alpha}{2} \sum_{j=1}^{d} w_j^2$$

$$w_j = \frac{\rho_j}{z_j + \alpha}$$

$\rho_j$ =  (Residual Correlation)
$z_j$ =  $j$   (Squared sum)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def coordinate_descent_ridge(X, y, alpha=0.1, num_iters=100):
    m, n = X.shape
    w = np.zeros(n)

    for iteration in range(num_iters):
        for j in range(n):
            # Feature j predictions
            X_j = X[:, j]
            y_pred_without_j = X @ w - (w[j] * X_j)


            rho_j = X_j.T @ (y - y_pred_without_j)
            z_j = np.sum(X_j ** 2)

            # Ridge Update Rule (Soft thresholding )
            w[j] = rho_j / (z_j + alpha * m)

    return w

np.random.seed(42)
X = np.random.randn(100, 3)
true_w = np.array([3, -2, 0])
y = X @ true_w + np.random.randn(100) * 0.5

# 2. Ridge Regularization Path
alphas_ridge = np.linspace(0.01, 100.0, 100) # Ridge
weights_path_ridge = []

for a in alphas_ridge:
    w = coordinate_descent_ridge(X, y, alpha=a, num_iters=50)
    weights_path_ridge.append(w.copy())

weights_path_ridge = np.array(weights_path_ridge)

# 3.
plt.figure(figsize=(10, 6))
for j in range(weights_path_ridge.shape[1]):
    plt.plot(alphas_ridge, weights_path_ridge[:, j], linewidth=2,
             label=f'Feature {j+1} (True W: {true_w[j]})')

plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.title("Ridge Regularization Path (Alpha vs. Weights)", fontsize=14)
plt.xlabel("Alpha (Penalty Strength)", fontsize=12)
plt.ylabel("Coefficient / Weight Value", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# New Section

# What is Elastic Net?
Elastic Net is a hybrid regularization technique that combines the best of both Lasso (L1) and Ridge (L2).Lasso (L1) is great at feature selection (pushing useless weights to exactly zero) but can be unstable if two features are highly correlated (it randomly picks one and drops the other).
2. Ridge (L2) is great at stability (handling correlated features smoothly) but never pushes weights entirely to zero.
3. Elastic Net uses both penalties at the same time. It forces useless features to zero (like Lasso) while maintaining stability and grouping correlated features together (like Ridge).
1. Mathematical FormulaIn Elastic Net, we introduce an overall penalty strength ($\alpha$) and a mixing parameter called l1_ratio (often denoted as $\rho$).If l1_ratio = 1.0, it is 100% Lasso.If l1_ratio = 0.0, it is 100% Ridge.The cost function is:$$J(w) = \frac{1}{2n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \cdot \rho \sum_{j=1}^{d} \vert{}w_j\vert{} + \frac{\alpha \cdot (1 - \rho)}{2} \sum_{j=1}^{d} w_j^2$$Coordinate Descent Update Rule:Because Elastic Net uses both penalties, the update rule for a single weight $w_j$ elegantly combines the Soft Thresholding of Lasso and the Division of Ridge:$$w_j = \frac{\text{SoftThreshold}(\rho_j, \lambda_1)}{z_j + \lambda_2}$$Where:$\lambda_1 = \alpha \times \rho \times m$ (The L1 penalty part)$\lambda_2 = \alpha \times (1 - \rho) \times m$ (The L2 penalty part)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# We redefine soft_threshold just to be safe
def soft_threshold(rho, penalty):
    if rho < -penalty:
        return rho + penalty
    elif rho > penalty:
        return rho - penalty
    else:
        return 0

def coordinate_descent_elastic_net(X, y, alpha=0.1, l1_ratio=0.5, num_iters=100):
    m, n = X.shape
    w = np.zeros(n)

    # Calculate the separate L1 and L2 penalty terms
    l1_penalty = alpha * l1_ratio * m
    l2_penalty = alpha * (1 - l1_ratio) * m

    for iteration in range(num_iters):
        for j in range(n):
            # Calculate predictions without feature j
            X_j = X[:, j]
            y_pred_without_j = X @ w - (w[j] * X_j)

            # Calculate correlation (rho) and normalizer (z)
            rho_j = X_j.T @ (y - y_pred_without_j)
            z_j = np.sum(X_j ** 2)

            # Elastic Net Update: Soft Thresholding (L1) divided by (z + L2 penalty)
            if z_j + l2_penalty == 0:
                w[j] = 0
            else:
                w[j] = soft_threshold(rho_j, l1_penalty) / (z_j + l2_penalty)

    return w

# 1. Generate the same synthetic data
np.random.seed(42)
X = np.random.randn(100, 3)
true_w = np.array([3, -2, 0])
y = X @ true_w + np.random.randn(100) * 0.5

# 2. Plot Elastic Net Regularization Path
alphas_enet = np.linspace(0.01, 10.0, 100)
weights_path_enet = []
l1_ratio_val = 0.5 # 50% Lasso, 50% Ridge balance

for a in alphas_enet:
    w = coordinate_descent_elastic_net(X, y, alpha=a, l1_ratio=l1_ratio_val, num_iters=50)
    weights_path_enet.append(w.copy())

weights_path_enet = np.array(weights_path_enet)

# 3. Plot the Graph
plt.figure(figsize=(10, 6))
for j in range(weights_path_enet.shape[1]):
    plt.plot(alphas_enet, weights_path_enet[:, j], linewidth=2,
             label=f'Feature {j+1} (True W: {true_w[j]})')

plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.title(f"Elastic Net Regularization Path (l1_ratio={l1_ratio_val})", fontsize=14)
plt.xlabel("Alpha (Overall Penalty Strength)", fontsize=12)
plt.ylabel("Coefficient / Weight Value", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()